In [1]:
import glob
import json
import numpy as np
import os
import pandas as pd
import re
import seaborn as sns

from matplotlib import pyplot as plt
from pathlib import Path
from typing import Dict, List

from graphix.aggregators import CombineAggregator, SummationAggregator
from graphix.base_types import Node
from graphix.text_and_config_interface import parse_config_json, parse_stats_txt

Matplotlib is building the font cache; this may take a moment.


ModuleNotFoundError: No module named 'graphix'

In [ ]:
def get_root_and_stat(stats_path, config_path, name: str, index: Dict, version: int):
    root = None
    stat = None
    with open(config_path, "r") as config_file:
        config_json = json.load(config_file)
        root = Node(name, "")
        parse_config_json(root, config_json)
        stat = parse_stats_txt(index, root, stats_path, version)
    return root, stat

In [ ]:
def search_children(root: Node, patterns: Dict[str, str]) -> Dict[str, List[Node]]:
    ret = dict()
    for key in patterns:
        ret[key] = []

    frontier = [root]
    next_frontier = []
    while frontier:
        for node in frontier:
            for child in node.children():
                for key, pattern in patterns.items():
                    if re.match(pattern, child.path()):
                        ret[key].append(child)

                next_frontier.append(child)
        frontier = next_frontier
        next_frontier = []

    return ret

In [ ]:
def plot_distribution_pdf(fig, ax, dist, parent):
    bucket_intervals = dist.get_buckets_for_parent(parent)
    bucket_freqs = dist.get_freqs_for_parent(parent)
    total_count = sum(bucket_freqs)
    pdf_values = []
    bar_centers = []
    bar_widths = []
    for (start_i, end_i), count in zip(bucket_intervals, bucket_freqs):
        width = end_i - start_i
        if width > 0:
            pdf_val = count / (total_count * width)
        else:
            pdf_val = 0.0
        pdf_values.append(pdf_val)
        bar_centers.append((start_i + end_i) / 2.0)
        bar_widths.append(width if width > 0 else 0)
    ax.bar(bar_centers, pdf_values, width=bar_widths, align="center", edgecolor="black")

def plot_distribution_cdf(fig, ax, dist, parent):
    bucket_intervals = dist.get_buckets_for_parent(parent)
    bucket_freqs = dist.get_freqs_for_parent(parent)
    total_count = sum(bucket_freqs)
    cdf_counts = np.cumsum(bucket_freqs)
    cdf_values = cdf_counts / total_count
    bar_centers = []
    bar_widths = []
    bar_heights = []
    for i, (start_i, end_i) in enumerate(bucket_intervals):
        bar_centers.append((start_i + end_i) / 2.0)
        width = end_i - start_i
        if width <= 0:
            width = 0
        bar_widths.append(width)
        bar_heights.append(cdf_values[i])
    ax.bar(bar_centers, bar_heights, width=bar_widths, align="center", edgecolor="black")



In [ ]:
experiment_path = Path("/home/msamani/gem5-out/scatter-gather-opt/profiling")

roots = []
stats = []
for root_path, _, files in os.walk(experiment_path):
    if "stats.txt" not in files:
        continue
    files = glob.glob("stats_dump_*", root_dir=root_path)
    if not files:
        continue
    latest_file = max(files, key=lambda x: int(re.search(r'stats_dump_(\d+)', x).group(1)))

    stats_path = Path(root_path) / latest_file
    config_path = Path(root_path) / "config.json"
    constructor_path = Path(root_path) / "constructor.py"
    with open(constructor_path, "r") as constuctor:
        lines = [line.strip().replace(",", "").replace("\"", "").replace("workload", "app") for line in constuctor.readlines()[6:-4]]
    workload = lines[0]
    workload_params = "::".join(lines[1:]).replace("_name", "").replace("num_processes", "np")
    workload_params = re.sub(r"dim_x=(\d+)::dim_y=\1::dim_z=\1", r"dim=\1", workload_params)
    name = f"{workload}::{workload_params}"
    index = {"workload": workload, "workload_params": workload_params}
    try:
        root, stat = get_root_and_stat(stats_path, config_path, name, index, -1)
        roots.append(root)
        stats.append(stat)
    except IndexError:
        print(f"Index error in {root_path}")
        continue
    except ValueError:
        print(f"Value error in {root_path}")
        continue


In [ ]:
with open("test.json", "w") as dump:
    json.dump(stats, dump, indent=2, default=str)

In [ ]:
combine = CombineAggregator()
summation = SummationAggregator()

In [ ]:
sequencers = []
l1d_caches = []
l2_caches = []
system_caches = []
for root in roots:
    children = search_children(root, {"sequencer": r"^board.cache_hierarchy.core_tiles(\d+).core_clusters(\d+).dcache.sequencer$",
                                      "dcache": r"^board.cache_hierarchy.core_tiles(\d+).core_clusters(\d+).dcache.cache$",
                                      "l2cache": r"^board.cache_hierarchy.core_tiles(\d+).core_clusters(\d+).l2cache.cache$",
                                      "system_cache": r"^board.cache_hierarchy.system_caches(\d+)$"})
    sequencers.append(children["sequencer"])
    l1d_caches.append(children["dcache"])
    l2_caches.append(children["l2cache"])
    system_caches.append(children["system_cache"])

In [ ]:
arrays = ["global_p", "gradzatp_point_volume", "global_point_gradient", "global_z", "global_zone_field", "other"]
# arrays = ["global_p", "gradzatp_point_volume", "global_point_gradient", "other"]

fig, ax_group = plt.subplots(nrows=2, ncols=3, figsize=(12, 7.5), sharey=True)

for index, pack in enumerate(zip(roots, stats, sequencers, l1d_caches, l2_caches, system_caches)):
    for arr_index, array in enumerate(arrays):
        root, stat, sequencers, l1d_caches, l2_caches, system_caches = pack
        p_useful = stat[f"usefulBytes_{array}"].filter_parents(these_parents=set(l1d_caches), not_these_parents=set()).aggregate_using(combine)
        plot_distribution_cdf(fig, ax_group[arr_index//3][arr_index%3], p_useful, combine)
        ax_group[arr_index//3][arr_index%3].set_title(f"data_structure: {'_'.join(array.split('_')[1:]) if array != 'other' else 'other'}")
        ax_group[arr_index//3][arr_index%3].set_xlim(0, 65)
    break

fig.savefig("structures_cdf.png", format="png", dpi=100)


In [ ]:
fig, ax_group = plt.subplots(nrows=1, ncols=3, figsize=(13.5, 4), sharey=True)

for index, pack in enumerate(zip(local_dcaches, local_roots, local_stats)):
    my_l1d_caches, root, stat = pack
    cache_line_size = root.board.cache_line_size
    useful_bytes = stat["usefulBytes"]
    l1d_useful_bytes = useful_bytes.filter_parents(these_parents=set(my_l1d_caches), not_these_parents=set())
    l1d_useful_bytes = l1d_useful_bytes.aggregate_using(combine)
    plot_distribution_cdf(fig, ax_group[index], l1d_useful_bytes, combine)
    ax_group[index].set_title(f"{l1d_useful_bytes.index()["workload"].upper()},{l1d_useful_bytes.index()["input"].upper()}")
    ax_group[index].set_xlim((0, 65))
    ax_group[index].set_xticks(range(0, 65, 8))
    ax_group[index].axvline(cache_line_size + 1, color="red", linestyle="--")

ax_group[0].set_ylabel("Cummulative Frequency")
fig.text(0.5, 0, "Number of Accesses Bytes from Cache Block", fontdict={"size":12})

fig.savefig("l1d_useful_bytes.png", format="png", dpi=72)

In [ ]:
local_pack = [(current_l1d_caches, current_l2_caches, current_root, current_stats) for current_l1d_caches, current_l2_caches, current_root, current_stats in zip(l1d_caches, l2_caches, roots, stats) if current_stats["simSeconds"].index()["mod"] == "no_prefetcher"]
local_dcaches, local_l2caches, local_roots, local_stats = zip(*local_pack)

In [ ]:
fig, ax_group = plt.subplots(nrows=1, ncols=5, figsize=(24, 4))


for index, pack in enumerate(zip(local_dcaches, local_roots, local_stats)):
    my_l1d_caches, root, stat = pack
    cache_line_size = root.board.cache_line_size
    useful_bytes = stat["usefulBytes"]
    l1d_useful_bytes = useful_bytes.filter_parents(these_parents=set(my_l1d_caches), not_these_parents=set())
    l1d_useful_bytes = l1d_useful_bytes.aggregate_using(combine)
    plot_distribution_cdf(fig, ax_group[index], l1d_useful_bytes, combine)
    ax_group[index].set_title(f"Workload: {l1d_useful_bytes.index()["workload"]}, Input: {l1d_useful_bytes.index()["input"]}")
    ax_group[index].set_xlim((0, 65))
    ax_group[index].set_xticks(range(0, 65, 8))
    ax_group[index].axvline(cache_line_size + 1, color="red", linestyle="--")

fig.savefig("l1d_useful_bytes_no_prefetcher.png", format="png", dpi=72)


In [ ]:
local_pack = [(current_l1d_caches, current_l2_caches, current_root, current_stats) for current_l1d_caches, current_l2_caches, current_root, current_stats in zip(l1d_caches, l2_caches, roots, stats) if current_stats["simSeconds"].index()["mod"] == "block_size_32"]
local_dcaches, local_l2caches, local_roots, local_stats = zip(*local_pack)

In [ ]:
fig, ax_group = plt.subplots(nrows=1, ncols=4, figsize=(24, 4))


for index, pack in enumerate(zip(local_dcaches, local_roots, local_stats)):
    my_l1d_caches, root, stat = pack
    cache_line_size = root.board.cache_line_size
    useful_bytes = stat["usefulBytes"]
    l1d_useful_bytes = useful_bytes.filter_parents(these_parents=set(my_l1d_caches), not_these_parents=set())
    l1d_useful_bytes = l1d_useful_bytes.aggregate_using(combine)
    plot_distribution_cdf(fig, ax_group[index], l1d_useful_bytes, combine)
    ax_group[index].set_title(f"Workload: {l1d_useful_bytes.index()["workload"]}, Input: {l1d_useful_bytes.index()["input"]}")
    ax_group[index].set_xlim((0, 64))
    ax_group[index].set_xticks(range(0, 64, 8))
    ax_group[index].axvline(cache_line_size + 1, color="red", linestyle="--")

fig.savefig("l1d_useful_bytes_32b_block.png", format="png", dpi=72)
